# Module 2 — Data Preprocessing (Google Colab mirror)

**Explainable Multilingual Fake News Detection using XLM-RoBERTa, Supervised Contrastive Learning, Fact Verification, and Explainable AI for Low-Resource Languages**

Mirrors the local Module 2 pipeline exactly — imports and runs the same
`src/preprocessing/*` code and `scripts/preprocess_dataset.py` entry point that
run locally, so there is only one implementation of the cleaning/verification/
split logic to keep correct.

**What this notebook does, in order:**
1. Mounts Google Drive.
2. Locates the project folder inside Drive (so `datasets/processed/`,
   `outputs/module2/`, and `logs/` are all written straight to Drive).
3. Installs dependencies.
4. Confirms Module 1's output (`datasets/processed/module1_dataset.csv`) is
   present — this notebook does not redo Module 1.
5. Runs the full Module 2 preprocessing pipeline.
6. Prints and verifies the final statistics, split counts, and a leakage check.

**Does not execute automatically on open** — run each cell yourself.

**GPU note:** Module 2 is pure CPU work (pandas/regex/BeautifulSoup) — no GPU
needed. Set the runtime to CPU.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Locate the project folder

Edit `DRIVE_PROJECT_PATH` if you placed the folder somewhere other than
`My Drive/Fake_News_Detection`.

In [ ]:
import os
import sys
from pathlib import Path

DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Fake_News_Detection"

PROJECT_ROOT = Path(DRIVE_PROJECT_PATH)
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found at {PROJECT_ROOT}. "
        "Upload/sync the Fake_News_Detection folder to this path in Google Drive, "
        "or edit DRIVE_PROJECT_PATH above."
    )

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Working directory:", os.getcwd())


## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt


## 4. Confirm Module 1 output is present

Module 2 reads `datasets/processed/module1_dataset.csv` — it does not
regenerate it. If this is missing, run the Module 1 notebook first.

In [ ]:
m1_path = PROJECT_ROOT / "datasets" / "processed" / "module1_dataset.csv"
if not m1_path.exists():
    raise FileNotFoundError(
        f"{m1_path} not found. Run notebooks/module1_colab.ipynb (or the local "
        "Module 1 pipeline) first, and make sure its output is synced to this "
        "Drive folder."
    )
import pandas as pd
_check = pd.read_csv(m1_path, low_memory=False)
print(f"Found Module 1 output: {len(_check)} rows.")
del _check


## 5. Run the Module 2 preprocessing pipeline

Cleans text (HTML/URL removal, Tamil-safe Unicode NFC normalization,
whitespace/text normalization), verifies language and label tags, analyzes
document length, removes invalid/too-short documents, re-checks for
duplicates post-cleaning, runs final quality checks, and produces a
reproducible train/validation/test split stratified jointly by language and
label.

In [ ]:
!python scripts/preprocess_dataset.py


## 6. Verify outputs

In [ ]:
import json

reports_dir = PROJECT_ROOT / "outputs" / "module2"
with open(reports_dir / "preprocessing_statistics.json", encoding="utf-8") as f:
    stats = json.load(f)

print("=== Cleaned dataset summary ===")
print(json.dumps(stats["stages"]["cleaned_dataset_summary"], indent=2, ensure_ascii=False))

print("\n=== Split ===")
split = stats["stages"]["split"]
print(f"train={split['n_train']}  val={split['n_val']}  test={split['n_test']}")
print("Leakage check:", split["leakage_check"])

print("\n=== Final quality checks ===")
print("Passed:", stats["stages"]["final_quality_checks"]["passed"])

processed_dir = PROJECT_ROOT / "datasets" / "processed"
print("\nFiles in datasets/processed/:")
for p in sorted(processed_dir.glob("*")):
    print(" -", p.name, f"({p.stat().st_size:,} bytes)")


## Done

Module 2 is complete once the cells above run without error and the leakage
check and quality checks both pass. `datasets/processed/{train,validation,test}.csv`
are what Module 3 (embeddings) will read from — do not begin Module 3 until
you've reviewed `outputs/module2/preprocessing_report.md` and given explicit
approval, per this project's module-by-module workflow.